In [ ]:
# I trained this model on googlecolab , and the  data folder is on my drive ,

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from torchvision import datasets,transforms
from torch.utils.data import DataLoader,random_split

In [ ]:
dataset_path = "/content/drive/MyDrive/kagglecatsanddogs_3367a/PetImages"

In [ ]:
# preprocess
transform = transforms.Compose([
    transforms.Resize((128,128)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))
])
# datasets
dataset = datasets.ImageFolder(
    root = dataset_path ,
    transform = transform
    
)

# Check Classes


print(dataset.classes)
print(dataset.class_to_idx)
# Split Dataset


train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size

train_dataset, test_dataset = random_split(
    dataset,
    [train_size, test_size]
)
# dataloader 
train_loader  = DataLoader(train_dataset,batch_size = 64 , shuffle = True)
test_loader = DataLoader(test_dataset,batch_size = 64)

In [ ]:
import torch.nn as nn
# building cnn 
import torch
import torch.nn as nn

class CNN(nn.Module):

    def __init__(self):

        super(CNN, self).__init__()

        # Convolution Layers
        self.conv_layers = nn.Sequential(

            # Conv Layer 1
            nn.Conv2d(
                in_channels=3,
                out_channels=32,
                kernel_size=3,
                padding=1
            ),

            nn.ReLU(),

            nn.MaxPool2d(
                kernel_size=2,
                stride=2
            ),

            # Conv Layer 2
            nn.Conv2d(
                in_channels=32,
                out_channels=64,
                kernel_size=3,
                padding=1
            ),

            nn.ReLU(),

            nn.MaxPool2d(
                kernel_size=2,
                stride=2
            )
        )

        # Fully Connected Layers
        self.fc_layers = nn.Sequential(

            nn.Flatten(),

            nn.Linear(
                64 * 32 * 32,
                128
            ),

            nn.ReLU(),

            nn.Linear(
                128,
                2
            )
        )

    def forward(self, x):

        x = self.conv_layers(x)

        x = self.fc_layers(x)

        return x

In [ ]:
model = CNN()
import torch.optim as optim
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

# Define device to use GPU if available, else CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device) # Move model to the selected device
print(f"Model moved to {device}")

model.train()
epochs = 20
for epoch in range(epochs):
  training_loss = 0
  for imgs ,labels in train_loader:
    imgs, labels = imgs.to(device), labels.to(device) # Move data to the selected device
    optimizer.zero_grad()
    output = model.forward(imgs)
    loss = criterion(output,labels)
    loss.backward()
    optimizer.step()
    training_loss += loss.item()
  print(f"per epoch training loss for epoch {epoch}/{epochs} :{training_loss/len(train_loader)}")

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval() # Set the model to evaluation mode
correct = 0
total = 0
with torch.no_grad(): # Disable gradient calculation during evaluation
    for imgs, labels in test_loader:
        imgs, labels = imgs.to(device), labels.to(device) # Move data to GPU
        outputs = model(imgs)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total
print(f'Accuracy of the model on the {total} test images: {accuracy:.2f}%')

#getting 76.34 Accuracy .
